# Production ML Infrastructure Tutorial

**Experiment tracking, hyperparameter tuning, and model registry**

This tutorial walks through the production ML tools in QuantStrata:

1. **Experiment tracking** – In-memory, MLflow, and Weights & Biases
2. **Hyperparameter tuning** – Search spaces, Optuna, and pruning
3. **Model registry** – Versioning and promotion to production
4. **Orchestrator pipeline** – Running tuning from config

**References:** `docs/reference/machine_learning/production_ml.md`, `docs/guides/machine_learning/experiment_tracking.md`, `hyperparameter_tuning.md`

---

## 1. Setup and imports

In [1]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
from pathlib import Path

np.random.seed(42)
print("Imports and path set.")

Imports and path set.


## 2. Experiment tracking

Use **InMemoryTracker** for notebooks and tests (no server). Log params, metrics, and artifacts.

In [2]:
from src.machine_learning.core.tracking import InMemoryTracker

tracker = InMemoryTracker(experiment_name="ml_production_tutorial")

with tracker.start_run("run_1"):
    tracker.log_params({"learning_rate": 0.001, "hidden_units": 128})
    for step in range(5):
        loss = 0.5 - step * 0.05 + np.random.randn() * 0.02
        tracker.log_metrics({"loss": float(loss)}, step=step)
    tracker.log_metrics({"final_loss": 0.25}, step=5)

runs = tracker.get_all_runs()
best = tracker.get_best_run("final_loss", minimize=True)
print(f"Runs: {len(runs)}")
print(f"Best run params: {best.params}")
print(f"Best final_loss: {best.metrics.get('final_loss')}")

Runs: 1
Best run params: {'learning_rate': 0.001, 'hidden_units': 128}
Best final_loss: [{'value': 0.25, 'step': 5, 'timestamp': '2026-02-04T17:31:17.485706'}]


## 3. Hyperparameter tuning with Optuna

Define a **SearchSpace**, an objective function, and run **run_optuna_tuning** with optional pruning.

In [3]:
from src.machine_learning.tuning import SearchSpace, MedianPruner, run_optuna_tuning

def dummy_objective(config, trial):
    """Minimize a simple function of lr and hidden_units; report intermediate for pruning."""
    lr = config["learning_rate"]
    hu = config["hidden_units"]
    loss = 0.1 * (np.log10(lr) + 3) ** 2 + 0.001 * (hu - 128) ** 2 / 1000
    for epoch in range(10):
        intermediate = loss * (1 - epoch / 15) + 0.02 * np.random.randn()
        trial.report(intermediate, epoch)
        if trial.should_prune():
            import optuna
            raise optuna.TrialPruned()
    return loss + 0.01 * np.random.randn()

space = (
    SearchSpace()
    .add_float("learning_rate", 1e-4, 1e-2, log=True)
    .add_int("hidden_units", 32, 256)
)

result = run_optuna_tuning(
    objective_fn=dummy_objective,
    search_space=space,
    n_trials=15,
    direction="minimize",
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=2),
    seed=42,
)

print(f"Best config: {result.best_config}")
print(f"Best score: {result.best_score:.6f}")
print(f"Completed trials: {result.n_completed}, Pruned: {result.n_pruned}")

/Users/joesstevens/PycharmProjects/QuantStrata/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-02-04 17:31:17,644] A new study created in memory with name: tuning_20260204_173117
Best trial: 4. Best value: 0.00420788:  47%|████▋     | 7/15 [00:00<00:00, 279.40it/s]

[I 2026-02-04 17:31:17,665] Trial 0 finished with value: 0.014362197419123843 and parameters: {'learning_rate': 0.0005611516415334506, 'hidden_units': 245}. Best is trial 0 with value: 0.014362197419123843.
[I 2026-02-04 17:31:17,668] Trial 1 finished with value: 0.01146253984065444 and parameters: {'learning_rate': 0.0029106359131330704, 'hidden_units': 166}. Best is trial 1 with value: 0.01146253984065444.
[I 2026-02-04 17:31:17,670] Trial 2 finished with value: 0.03145356905043012 and parameters: {'learning_rate': 0.0002051338263087451, 'hidden_units': 67}. Best is trial 1 with value: 0.01146253984065444.
[I 2026-02-04 17:31:17,674] Trial 3 pruned. 
[I 2026-02-04 17:31:17,677] Trial 4 finished with value: 0.004207875435768544 and parameters: {'learning_rate': 0.0015930522616241021, 'hidden_units': 191}. Best is trial 4 with value: 0.004207875435768544.
[I 2026-02-04 17:31:17,680] Trial 5 pruned. 
[I 2026-02-04 17:31:17,683] Trial 6 pruned. 
[I 2026-02-04 17:31:17,685] Trial 7 pruned

Best trial: 11. Best value: -0.0068468: 100%|██████████| 15/15 [00:00<00:00, 227.00it/s]

[I 2026-02-04 17:31:17,691] Trial 8 pruned. 
[I 2026-02-04 17:31:17,694] Trial 9 pruned. 
[I 2026-02-04 17:31:17,701] Trial 10 finished with value: 0.0032420090647011703 and parameters: {'learning_rate': 0.0018471184590376983, 'hidden_units': 193}. Best is trial 10 with value: 0.0032420090647011703.
[I 2026-02-04 17:31:17,708] Trial 11 finished with value: -0.006846803520549324 and parameters: {'learning_rate': 0.0015320252245209073, 'hidden_units': 194}. Best is trial 11 with value: -0.006846803520549324.
[I 2026-02-04 17:31:17,714] Trial 12 pruned. 
[I 2026-02-04 17:31:17,720] Trial 13 finished with value: 0.004271415789926537 and parameters: {'learning_rate': 0.0016692757600614092, 'hidden_units': 120}. Best is trial 11 with value: -0.006846803520549324.
[I 2026-02-04 17:31:17,726] Trial 14 finished with value: 0.018737119966176167 and parameters: {'learning_rate': 0.0013838713864867114, 'hidden_units': 34}. Best is trial 11 with value: -0.006846803520549324.
Best config: {'learning

## 4. Model registry

Register a model version with metadata and promote it to staging or production.

In [ ]:
from src.machine_learning.registry import ModelRegistry, ModelStage

# Use a directory that exists (create a dummy artifact dir for the tutorial)
artifact_dir = Path("./artifacts/ml_tutorial_pricer")
artifact_dir.mkdir(parents=True, exist_ok=True)
(artifact_dir / "model_info.json").write_text('{"name": "tutorial_pricer"}')

registry = ModelRegistry()  # or create_registry(base_path=...)
version = registry.register_model(
    name="option_pricer",
    model_path=artifact_dir,
    params={"framework": "keras", "input_dim": 10},
    metrics={"val_loss": 0.01},
    description="Tutorial v1",
)
print(f"Registered: {version} -> stage {version.stage}")

registry.promote_to_stage("option_pricer", version.version, ModelStage.STAGING)
versions = registry.list_versions("option_pricer")
print(f"After promote: versions = {[(v.version, v.stage) for v in versions]}")

TypeError: ModelRegistry.__init__() missing 1 required positional argument: 'storage_path'

## 5. Running the hyperparameter tuning pipeline

The orchestrator pipeline `ml.hyperparameter_tuning` is built with **create_hyperparameter_tuning_pipeline** (config + objective). For a config-driven run from YAML/CLI, use the example script **examples/pipelines/run_hyperparameter_tuning.py**.

Here we only show the pattern: build pipeline with a small search space and run it via PipelineRunner.

In [ ]:
# Optional: run the tuning pipeline (requires HyperparameterTuningConfig + objective)
# from src.orchestrator.pipelines.ml.hyperparameter_tuning import (
#     create_hyperparameter_tuning_pipeline,
#     HyperparameterTuningConfig,
# )
# config = HyperparameterTuningConfig(
#     search_space_config={...},
#     n_trials=20,
#     direction="minimize",
# )
# pipeline = create_hyperparameter_tuning_pipeline(config, objective_fn=my_objective)
# Then run with PipelineRunner and Context.

print("See examples/pipelines/run_hyperparameter_tuning.py for a full pipeline example.")

---
## Summary

- **Experiment tracking:** Use `InMemoryTracker` in notebooks; use `MLflowTracker` or `WandBTracker` for shared experiments.
- **Hyperparameter tuning:** Define `SearchSpace`, pass an objective to `run_optuna_tuning`, optionally with pruners.
- **Model registry:** Register artifacts with `ModelRegistry`, promote versions to staging/production.
- **Next:** Integrate tracking and registry into your training scripts and orchestrator pipelines.